# 04_01 — IT Filtering

Notebook này xác định các cặp Anh–Vi có thể dùng cho corpus IT sau Phase 03. Dữ liệu RAW và interim chỉ được đọc. Audit flag được xử lý theo policy versioned: cờ lỗi/noise bị auto-reject; language, extreme length ratio và very-long pair vào `manual_review`.

Chạy `Restart Kernel → Run All` cho từng `SOURCE_SHORT_NAME`. Lần chạy đầu tạo template `manual_review_decisions.csv`; điền các quyết định trong file đó rồi chạy lại để áp dụng.

In [1]:
import hashlib
import json
import os
import re
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

print('Python:', sys.version)
print('pandas:', pd.__version__)
print('Working directory:', os.getcwd())

Python: 3.14.6 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:29:05) [MSC v.1942 64 bit (AMD64)]
pandas: 3.0.5
Working directory: C:\Users\ADMIN\ENVI-IT-MT\notebooks\04_it_filtering


In [2]:
CURRENT_DIR = Path.cwd().resolve()


def find_project_root(start_path: Path) -> Path:
    for path in [start_path, *start_path.parents]:
        if (path / 'data').is_dir() and (path / 'notebooks').is_dir():
            return path
    raise FileNotFoundError('Không tìm thấy project root chứa data/ và notebooks/.')


PROJECT_ROOT = find_project_root(CURRENT_DIR)
print('Project root:', PROJECT_ROOT)

Project root: C:\Users\ADMIN\ENVI-IT-MT


## 1. Cấu hình policy

`envitech_reasoning` phải có evidence keyword để được approve tự động vì RAW nguồn này chứa cả dữ liệu không thuộc IT. Các nguồn còn lại là corpus công nghệ/localization; source prior chỉ áp dụng khi không có keyword. Không source prior nào được vượt qua audit flag.

In [3]:
SOURCE_SHORT_NAME = 'kde4'  # đổi để chạy nguồn khác

SUPPORTED_SOURCES = [
    'envitech_reasoning', 'tech_viet_translation', 'gnome', 'ubuntu', 'kde4',
]
IT_FILTERING_RULES_VERSION = '1.0.0'
MANUAL_DECISION_VALUES = {'approve', 'reject'}
AUTO_REJECT_AUDIT_FLAGS = {
    'blank_or_null', 'encoding_issue', 'identical_pair', 'exact_duplicate',
    'noise_url', 'noise_markup_or_html_entity', 'noise_windows_or_unix_file_path',
    'noise_placeholder', 'noise_ui_fragment',
}
MANUAL_REVIEW_AUDIT_FLAGS = {
    'language_suspect', 'extreme_length_ratio', 'very_long_pair',
}
assert AUTO_REJECT_AUDIT_FLAGS.isdisjoint(MANUAL_REVIEW_AUDIT_FLAGS)

# Không dùng source prior cho EnVi-Tech: dataset này có cả nội dung ngoài IT.
SOURCE_PRIOR_SUBDOMAIN = {
    'tech_viet_translation': 'documentation',
    'gnome': 'ui_localization',
    'ubuntu': 'ui_localization',
    'kde4': 'ui_localization',
}

# Keyword chỉ là bằng chứng tái lập được, không phải language/semantic classifier.
IT_TAXONOMY = {
    'coding': ['code', 'coding', 'program', 'programming', 'script', 'function', 'class', 'variable', 'compiler', 'debug', 'api', 'sdk', 'python', 'java', 'javascript', 'typescript', 'lập trình', 'mã nguồn', 'hàm', 'biến', 'gỡ lỗi'],
    'ai_ml': ['artificial intelligence', 'machine learning', 'deep learning', 'neural', 'model', 'dataset', 'training', 'inference', 'ai', 'ml', 'học máy', 'trí tuệ nhân tạo', 'mô hình'],
    'hardware': ['hardware', 'cpu', 'gpu', 'ram', 'memory', 'disk', 'ssd', 'driver', 'device', 'firmware', 'phần cứng', 'bộ nhớ', 'ổ đĩa', 'thiết bị'],
    'system': ['linux', 'windows', 'unix', 'kernel', 'server', 'network', 'system', 'process', 'service', 'package', 'operating system', 'hệ thống', 'máy chủ', 'mạng', 'gói phần mềm'],
    'ui_localization': ['window', 'dialog', 'menu', 'button', 'widget', 'screen', 'display', 'settings', 'option', 'user interface', 'cửa sổ', 'hộp thoại', 'trình đơn', 'nút', 'giao diện', 'tùy chọn', 'thiết lập'],
    'command_cli': ['command', 'terminal', 'shell', 'bash', 'cli', 'command line', 'install', 'uninstall', 'sudo', 'lệnh', 'dòng lệnh', 'cài đặt', 'gỡ cài đặt'],
    'documentation': ['documentation', 'document', 'manual', 'guide', 'readme', 'reference', 'tutorial', 'hướng dẫn', 'tài liệu', 'tham khảo'],
}

assert SOURCE_SHORT_NAME in SUPPORTED_SOURCES
INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim' / SOURCE_SHORT_NAME
IT_CORPUS_DIR = PROJECT_ROOT / 'data' / 'it_corpus' / SOURCE_SHORT_NAME
CLEANED_PARQUET_PATH = INTERIM_DIR / 'cleaned_pairs.parquet'
CLEANING_REPORT_PATH = INTERIM_DIR / 'cleaning_report.json'
CLEANING_MANIFEST_PATH = INTERIM_DIR / 'cleaning_manifest.json'
MANUAL_DECISIONS_PATH = IT_CORPUS_DIR / 'manual_review_decisions.csv'
IT_CORPUS_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
def sha256_file(file_path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with open(file_path, 'rb') as file:
        for chunk in iter(lambda: file.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def json_ready(value):
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if pd.isna(value):
        return None
    if hasattr(value, 'item'):
        return value.item()
    return value


def normalize_for_matching(value: str) -> str:
    return re.sub(r'\s+', ' ', value.casefold()).strip()


def keyword_subdomains(en_text: str, vi_text: str) -> list[str]:
    text = normalize_for_matching(f'{en_text} {vi_text}')
    return [
        subdomain for subdomain, keywords in IT_TAXONOMY.items()
        if any(keyword.casefold() in text for keyword in keywords)
    ]


for required_path in [CLEANED_PARQUET_PATH, CLEANING_REPORT_PATH, CLEANING_MANIFEST_PATH]:
    assert required_path.is_file(), f'Không tìm thấy input Phase 03: {required_path}'

with open(CLEANING_REPORT_PATH, 'r', encoding='utf-8') as file:
    cleaning_report = json.load(file)
with open(CLEANING_MANIFEST_PATH, 'r', encoding='utf-8') as file:
    cleaning_manifest = json.load(file)

assert cleaning_report['source_short_name'] == SOURCE_SHORT_NAME
assert cleaning_report['cleaning_schema_version'] == '1.1'
assert cleaning_manifest['source_short_name'] == SOURCE_SHORT_NAME
manifest_artifacts = {item['path'].replace('\\', '/'): item for item in cleaning_manifest['artifacts']}
cleaned_relative_path = str(CLEANED_PARQUET_PATH.relative_to(PROJECT_ROOT)).replace('\\', '/')
assert cleaned_relative_path in manifest_artifacts, 'Manifest Phase 03 không chứa cleaned_pairs.parquet.'
assert sha256_file(CLEANED_PARQUET_PATH) == manifest_artifacts[cleaned_relative_path]['sha256'], (
    'Checksum cleaned_pairs.parquet khác manifest. Hãy chạy lại Phase 03.'
)

pairs_df = pd.read_parquet(CLEANED_PARQUET_PATH)
required_columns = {'source_short_name', 'raw_row_index', 'en_clean', 'vi_clean', 'pair_sha256', 'audit_flags', 'requires_phase_04_review'}
assert required_columns.issubset(pairs_df.columns), f'Thiếu cột: {sorted(required_columns - set(pairs_df.columns))}'
assert pairs_df['source_short_name'].eq(SOURCE_SHORT_NAME).all()
assert pairs_df['raw_row_index'].is_unique
print('Loaded Phase 03 pairs:', len(pairs_df))

Loaded Phase 03 pairs: 39888


## 2. Phân loại tự động và hàng chờ review

Audit flag có ưu tiên cao nhất: auto-reject được áp dụng trước nếu một pair mang nhiều cờ. Các cờ `language_suspect`, `extreme_length_ratio`, `very_long_pair` mới tạo `manual_review`; cờ chưa biết cũng được review thủ công. Với source không có prior, thiếu evidence taxonomy bị `reject`; đây là cách tránh đưa toàn bộ EnVi-Tech Reasoning vào corpus IT chỉ dựa vào danh tính nguồn.

In [5]:
filtered_df = pairs_df.copy()
filtered_df['keyword_subdomains'] = [
    keyword_subdomains(en_text, vi_text)
    for en_text, vi_text in filtered_df[['en_clean', 'vi_clean']].itertuples(index=False, name=None)
]
filtered_df['it_subdomain'] = filtered_df['keyword_subdomains'].map(lambda values: ';'.join(values))
filtered_df['classification_method'] = 'keyword'
source_prior = SOURCE_PRIOR_SUBDOMAIN.get(SOURCE_SHORT_NAME)
prior_mask = filtered_df['it_subdomain'].eq('') & bool(source_prior)
filtered_df.loc[prior_mask, 'it_subdomain'] = source_prior
filtered_df.loc[prior_mask, 'classification_method'] = 'source_prior'

def split_audit_flags(value: object) -> list[str]:
    if pd.isna(value) or not str(value).strip():
        return []
    return [flag.strip() for flag in str(value).split(';') if flag.strip()]


filtered_df['audit_flag_list'] = filtered_df['audit_flags'].map(split_audit_flags)
filtered_df['auto_reject_audit_flags'] = filtered_df['audit_flag_list'].map(
    lambda flags: sorted(set(flags) & AUTO_REJECT_AUDIT_FLAGS)
)
filtered_df['manual_review_audit_flags'] = filtered_df['audit_flag_list'].map(
    lambda flags: sorted(set(flags) & MANUAL_REVIEW_AUDIT_FLAGS)
)
filtered_df['unknown_audit_flags'] = filtered_df['audit_flag_list'].map(
    lambda flags: sorted(set(flags) - AUTO_REJECT_AUDIT_FLAGS - MANUAL_REVIEW_AUDIT_FLAGS)
)

filtered_df['it_decision'] = 'approve'
filtered_df['decision_reason'] = 'it_taxonomy_evidence'
no_evidence_mask = filtered_df['it_subdomain'].eq('')
filtered_df.loc[no_evidence_mask, 'it_decision'] = 'reject'
filtered_df.loc[no_evidence_mask, 'decision_reason'] = 'no_it_taxonomy_evidence'
# Auto reject có ưu tiên cao nhất khi một pair mang nhiều audit flag.
auto_reject_mask = filtered_df['auto_reject_audit_flags'].map(bool)
filtered_df.loc[auto_reject_mask, 'it_decision'] = 'reject'
filtered_df.loc[auto_reject_mask, 'decision_reason'] = filtered_df.loc[auto_reject_mask, 'auto_reject_audit_flags'].map(
    lambda flags: 'audit_flag_auto_reject:' + ';'.join(flags)
)
manual_flag_mask = (
    ~auto_reject_mask
    & (filtered_df['manual_review_audit_flags'].map(bool) | filtered_df['unknown_audit_flags'].map(bool))
)
filtered_df.loc[manual_flag_mask, 'it_decision'] = 'manual_review'
filtered_df.loc[manual_flag_mask, 'decision_reason'] = filtered_df.loc[manual_flag_mask].apply(
    lambda row: 'audit_flag_manual_review:' + ';'.join(row['manual_review_audit_flags'] + row['unknown_audit_flags']),
    axis=1,
)
filtered_df['manual_review_note'] = pd.NA

assert filtered_df['it_decision'].isin({'approve', 'reject', 'manual_review'}).all()
display(filtered_df['it_decision'].value_counts().to_frame('count'))

,count
it_decision,
approve,33813
reject,4220
manual_review,1855


In [6]:
# File này là worksheet review: tạo sẵn tất cả pair mang audit flag và giữ quyết định qua các lần chạy.
MANUAL_DECISION_COLUMNS = ['raw_row_index', 'manual_decision', 'manual_it_subdomain', 'manual_note']
MANUAL_WORKSHEET_COLUMNS = [
    'raw_row_index', 'source_short_name', 'en_clean', 'vi_clean', 'audit_flags',
    'suggested_it_subdomain', 'suggested_reason', *MANUAL_DECISION_COLUMNS[1:],
]
review_candidates = filtered_df.loc[filtered_df['it_decision'].eq('manual_review'), [
    'raw_row_index', 'source_short_name', 'en_clean', 'vi_clean', 'audit_flags', 'it_subdomain', 'decision_reason'
]].copy()
review_candidates = review_candidates.rename(columns={
    'it_subdomain': 'suggested_it_subdomain',
    'decision_reason': 'suggested_reason',
})

if MANUAL_DECISIONS_PATH.is_file():
    existing_worksheet = pd.read_csv(MANUAL_DECISIONS_PATH, dtype='string')
    missing_manual_columns = set(MANUAL_DECISION_COLUMNS) - set(existing_worksheet.columns)
    assert not missing_manual_columns, f'Manual decisions thiếu cột: {sorted(missing_manual_columns)}'
    existing_decisions = existing_worksheet[MANUAL_DECISION_COLUMNS].copy()
    # pandas 3 + ArrowStringArray không hỗ trợ DataFrame.any(axis=1) trên string dtype.
    entered_mask = pd.Series(False, index=existing_decisions.index, dtype='bool')
    for column in MANUAL_DECISION_COLUMNS[1:]:
        entered_mask |= existing_decisions[column].fillna('').astype('string').str.strip().ne('').astype('bool')
    existing_decisions = existing_decisions.loc[entered_mask].copy()
else:
    existing_decisions = pd.DataFrame(columns=MANUAL_DECISION_COLUMNS)

if not existing_decisions.empty:
    existing_decisions['raw_row_index'] = pd.to_numeric(existing_decisions['raw_row_index'], errors='raise').astype('int64')
    assert existing_decisions['raw_row_index'].is_unique, 'Một row không được có hai manual decision.'
    assert set(existing_decisions['raw_row_index']).issubset(set(review_candidates['raw_row_index'])), (
        'manual_review_decisions.csv có raw_row_index không thuộc danh sách audit review hiện tại.'
    )
    assert existing_decisions['manual_decision'].dropna().isin(MANUAL_DECISION_VALUES).all(), (
        'manual_decision chỉ nhận approve hoặc reject.'
    )

# Bảo đảm hai phía của merge luôn cùng dtype, kể cả CSV trống/được Excel lưu lại.
review_candidates['raw_row_index'] = pd.to_numeric(review_candidates['raw_row_index'], errors='raise').astype('int64')
existing_decisions['raw_row_index'] = pd.to_numeric(existing_decisions['raw_row_index'], errors='raise').astype('int64')
manual_worksheet = review_candidates.merge(existing_decisions, on='raw_row_index', how='left', validate='one_to_one')
for column in MANUAL_DECISION_COLUMNS[1:]:
    manual_worksheet[column] = manual_worksheet[column].astype('string')
manual_worksheet = manual_worksheet[MANUAL_WORKSHEET_COLUMNS].sort_values('raw_row_index').reset_index(drop=True)
manual_worksheet.to_csv(MANUAL_DECISIONS_PATH, index=False, encoding='utf-8-sig')

manual_decisions = manual_worksheet[MANUAL_DECISION_COLUMNS].dropna(how='all').copy()
filtered_df = filtered_df.merge(manual_decisions, on='raw_row_index', how='left', validate='one_to_one')
apply_manual_mask = filtered_df['it_decision'].eq('manual_review') & filtered_df['manual_decision'].isin(MANUAL_DECISION_VALUES)
filtered_df.loc[apply_manual_mask, 'it_decision'] = filtered_df.loc[apply_manual_mask, 'manual_decision']
filtered_df.loc[apply_manual_mask, 'decision_reason'] = 'manual_review_decision'
manual_subdomain_mask = apply_manual_mask & filtered_df['manual_it_subdomain'].notna()
filtered_df.loc[manual_subdomain_mask, 'it_subdomain'] = filtered_df.loc[manual_subdomain_mask, 'manual_it_subdomain']
filtered_df.loc[apply_manual_mask, 'manual_review_note'] = filtered_df.loc[apply_manual_mask, 'manual_note']

manual_review_template = filtered_df.loc[filtered_df['it_decision'].eq('manual_review'), [
    'raw_row_index', 'source_short_name', 'en_clean', 'vi_clean', 'audit_flags', 'it_subdomain', 'decision_reason'
]].copy()
print('Review worksheet rows:', len(manual_worksheet))
print('Manual decisions applied:', int(filtered_df['decision_reason'].eq('manual_review_decision').sum()))
print('Manual review remaining:', len(manual_review_template))

Review worksheet rows: 1855
Manual decisions applied: 1855
Manual review remaining: 0


## 3. Lưu artifact và gate Phase 05

`approved_for_dataset_building` chỉ là `true` khi không còn cặp `manual_review`. Nếu còn review, artifact vẫn được lưu để review nhưng Phase 05 phải dừng.

In [7]:
OUTPUT_COLUMNS = [
    'source_short_name', 'raw_row_index', 'en_raw', 'vi_raw', 'en_clean', 'vi_clean',
    'pair_sha256', 'audit_flags', 'audit_flag_list', 'auto_reject_audit_flags',
    'manual_review_audit_flags', 'unknown_audit_flags', 'requires_phase_04_review', 'keyword_subdomains',
    'it_subdomain', 'classification_method', 'it_decision', 'decision_reason',
    'manual_review_note',
]
approved_pairs = filtered_df.loc[filtered_df['it_decision'].eq('approve'), OUTPUT_COLUMNS].sort_values('raw_row_index').reset_index(drop=True)
rejected_pairs = filtered_df.loc[filtered_df['it_decision'].eq('reject'), OUTPUT_COLUMNS].sort_values('raw_row_index').reset_index(drop=True)
manual_review_pairs = filtered_df.loc[filtered_df['it_decision'].eq('manual_review'), OUTPUT_COLUMNS].sort_values('raw_row_index').reset_index(drop=True)

APPROVED_PARQUET_PATH = IT_CORPUS_DIR / 'approved_it_pairs.parquet'
APPROVED_JSONL_PATH = IT_CORPUS_DIR / 'approved_it_pairs.jsonl'
REJECTED_PARQUET_PATH = IT_CORPUS_DIR / 'rejected_or_non_it_pairs.parquet'
MANUAL_REVIEW_PATH = IT_CORPUS_DIR / 'manual_review_pairs.csv'
IT_FILTERING_REPORT_PATH = IT_CORPUS_DIR / 'it_filtering_report.json'
IT_FILTERING_MANIFEST_PATH = IT_CORPUS_DIR / 'it_filtering_manifest.json'

approved_pairs.to_parquet(APPROVED_PARQUET_PATH, index=False)
approved_pairs.to_json(APPROVED_JSONL_PATH, orient='records', lines=True, force_ascii=False)
rejected_pairs.to_parquet(REJECTED_PARQUET_PATH, index=False)
manual_review_pairs.to_csv(MANUAL_REVIEW_PATH, index=False, encoding='utf-8-sig')

run_timestamp_utc = datetime.now(timezone.utc).isoformat()
approved_for_dataset_building = len(manual_review_pairs) == 0
it_filtering_report = {
    'it_filtering_schema_version': '1.0',
    'it_filtering_timestamp_utc': run_timestamp_utc,
    'source_short_name': SOURCE_SHORT_NAME,
    'rules_version': IT_FILTERING_RULES_VERSION,
    'input_cleaning_manifest_sha256': sha256_file(CLEANING_MANIFEST_PATH),
    'input_cleaned_pairs_sha256': sha256_file(CLEANED_PARQUET_PATH),
    'input_rows': int(len(filtered_df)),
    'decision_counts': {key: int(value) for key, value in filtered_df['it_decision'].value_counts().to_dict().items()},
    'subdomain_counts_approved': {key: int(value) for key, value in approved_pairs['it_subdomain'].value_counts().to_dict().items()},
    'manual_review_remaining': int(len(manual_review_pairs)),
    'audit_flag_action_policy': {
        'auto_reject': sorted(AUTO_REJECT_AUDIT_FLAGS),
        'manual_review': sorted(MANUAL_REVIEW_AUDIT_FLAGS),
        'unknown_flag_handling': 'manual_review',
    },
    'decision': {
        'approved_for_dataset_building': approved_for_dataset_building,
        'reason': 'all_manual_review_resolved' if approved_for_dataset_building else 'manual_review_remaining',
    },
}
with open(IT_FILTERING_REPORT_PATH, 'w', encoding='utf-8') as file:
    json.dump(json_ready(it_filtering_report), file, ensure_ascii=False, indent=2, allow_nan=False)

output_files = [APPROVED_PARQUET_PATH, APPROVED_JSONL_PATH, REJECTED_PARQUET_PATH, MANUAL_REVIEW_PATH, IT_FILTERING_REPORT_PATH]
it_filtering_manifest = {
    'manifest_schema_version': '1.0',
    'created_at_utc': run_timestamp_utc,
    'source_short_name': SOURCE_SHORT_NAME,
    'rules_version': IT_FILTERING_RULES_VERSION,
    'cleaning_manifest_sha256': sha256_file(CLEANING_MANIFEST_PATH),
    'artifacts': [
        {'path': str(path.relative_to(PROJECT_ROOT)), 'bytes': int(path.stat().st_size), 'sha256': sha256_file(path)}
        for path in output_files
    ],
}
with open(IT_FILTERING_MANIFEST_PATH, 'w', encoding='utf-8') as file:
    json.dump(json_ready(it_filtering_manifest), file, ensure_ascii=False, indent=2, allow_nan=False)

print('Approved:', len(approved_pairs))
print('Rejected:', len(rejected_pairs))
print('Manual review:', len(manual_review_pairs))
print('Approved for Phase 05:', approved_for_dataset_building)

Approved: 33827
Rejected: 6061
Manual review: 0
Approved for Phase 05: True


In [8]:
# Verification: mọi input row có đúng một quyết định và chỉ các cặp approve được đưa sang Phase 05.
all_output_indexes = pd.concat([approved_pairs['raw_row_index'], rejected_pairs['raw_row_index'], manual_review_pairs['raw_row_index']])
assert len(all_output_indexes) == len(filtered_df)
assert all_output_indexes.nunique() == len(filtered_df)
assert set(all_output_indexes) == set(filtered_df['raw_row_index'])
assert approved_pairs['it_decision'].eq('approve').all()
assert rejected_pairs['it_decision'].eq('reject').all()
assert manual_review_pairs['it_decision'].eq('manual_review').all()
assert pd.read_parquet(APPROVED_PARQUET_PATH).shape[0] == len(approved_pairs)
assert all(path.is_file() for path in output_files + [IT_FILTERING_MANIFEST_PATH, MANUAL_DECISIONS_PATH])
print('Verification passed:', True)

Verification passed: True
